# AI Forex Bot v2 — Training 7 Tahun
Notebook untuk training model dengan data 7 tahun di Google Colab

## Sebelum mulai:
1. Upload folder `data/` (parquet) ke Google Drive → `MyDrive/forex-bot/data/`
2. Upload `.env` ke `MyDrive/forex-bot/.env`
3. (Opsional) Upload `models/` ke `MyDrive/forex-bot/models/`

Struktur Drive:
```
MyDrive/forex-bot/
├── .env
├── data/
│   └── EURUSD.fl/
│       ├── tf_5.parquet
│       ├── tf_15.parquet
│       └── tf_30.parquet
└── models/          (opsional)
```

---
## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## 2. Clone Repository

In [ ]:
!git clone https://github.com/snooters/bot-ai-forex-v2.git
%cd bot-ai-forex-v2

---
## 3. Copy Data & Config dari Drive

In [ ]:
import shutil, os
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/forex-bot'

# Copy .env
env_src = f'{DRIVE_BASE}/.env'
if os.path.exists(env_src):
    shutil.copy(env_src, '.env')
    print('✓ .env copied')
else:
    print('⚠ .env not found in Drive. Upload .env dulu!')

# Copy data parquet
data_src = f'{DRIVE_BASE}/data'
dst_data = 'data/historical'
if os.path.exists(data_src):
    for symbol_dir in os.listdir(data_src):
        sym_path = os.path.join(data_src, symbol_dir)
        if os.path.isdir(sym_path):
            dst_sym = os.path.join(dst_data, symbol_dir)
            os.makedirs(dst_sym, exist_ok=True)
            for f in os.listdir(sym_path):
                shutil.copy(os.path.join(sym_path, f), os.path.join(dst_sym, f))
            print(f'✓ {symbol_dir}: {len(os.listdir(sym_path))} files copied')
else:
    print('⚠ Data folder not found in Drive. Upload data dulu!')

# Copy existing models (optional)
models_src = f'{DRIVE_BASE}/models'
if os.path.exists(models_src):
    !cp -r "{models_src}/"* models/
    print('✓ Existing models restored')
else:
    print('• No existing models found (starting fresh)')

print('\n✅ Setup selesai')

---
## 4. Install Dependencies

In [ ]:
!pip install -r requirements.txt
print('\n✅ Dependencies installed')

---
## 5. Verifikasi Data
Pastikan data parquet terbaca dengan benar.

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path('data/historical')
if data_dir.exists():
    for parquet_file in sorted(data_dir.rglob('*.parquet')):
        df = pd.read_parquet(parquet_file)
        print(f'{parquet_file.name}: {len(df):,} rows | '
              f'{df["time"].min()} → {df["time"].max()}')
else:
    print('⚠ Folder data/historical tidak ditemukan!')

---
## 6. Jalankan Training 7 Tahun

Parameter yang akan dipakai:
- `--from-storage`: pakai data parquet yang sudah di-copy
- `--all`: training semua timeframe (M5, M15, M30)
- `--force`: bypass market check
- `--days 2555`: 7 tahun data (2555 hari)

⏱ Estimasi waktu: **15-45 menit** tergantung volume data & model (XGBoost, RF, LightGBM, LSTM)

In [ ]:
# Load .env config
from dotenv import load_dotenv
load_dotenv()

import os
print(f'HISTORICAL_YEARS={os.getenv("HISTORICAL_YEARS", "not set")}')
print(f'ROLLING_TRAINING_WINDOW_DAYS={os.getenv("ROLLING_TRAINING_WINDOW_DAYS", "not set")}')

print('\n🚀 Mulai training dengan 7 tahun data...\n')

In [ ]:
!python main.py train --from-storage --all --force --days 2555

---
## 7. Backup Model ke Google Drive

Simpan model hasil training dan metadata ke Drive agar tidak hilang.

In [ ]:
models_drive = '/content/drive/MyDrive/forex-bot/models'
os.makedirs(models_drive, exist_ok=True)

# Copy semua model baru
!cp -r models/* "{models_drive}/"

# Copy metadata training
if os.path.exists('models/retrain_counter.json'):
    shutil.copy('models/retrain_counter.json', models_drive)

print('\n✅ Models saved to Google Drive')
print(f'📁 {models_drive}')

---
## 8. Ringkasan

Cek hasil training:

In [ ]:
from pathlib import Path
import json

print('📊 Model versions produced:')
for model_dir in sorted(Path('models').glob('model_*')):
    if model_dir.is_dir():
        metadata = model_dir / 'metadata.json'
        perf = model_dir / 'performance.json'
        if metadata.exists():
            m = json.loads(metadata.read_text())
            print(f'  {m["version"]:20s} | created: {m["created_at"][:19]}')
        if perf.exists():
            p = json.loads(perf.read_text())
            print(f'  {"":>20s} | OOS: WR={p.get("win_rate","-")} PF={p.get("profit_factor","-")} Grade={p.get("grade","-")}')

print('\n✅ Training selesai! Sekarang kamu bisa:')
print('   • Copy models/* dari Google Drive ke project lokal kamu')
print('   • Jalankan bot dengan model baru: python main.py live')